# Phase 3 — Heterogeneous Graph Construction

## Objectives

- Construct User–Movie–Genre heterogeneous graphs.
- Create Train and Train+Validation snapshots.
- Add reverse edges for message passing.
- Validate graph integrity and per-user temporal isolation.
- Save graph artifacts for downstream GNN training.

**Environment:** Windows 11 / VS Code / Local Python `.venv`.

**Framework:** PyTorch Geometric.

**Scope:** Graph construction only; no model training.

Import và thiết lập đường dẫn

In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd
import torch
import torch_geometric
import matplotlib.pyplot as plt

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "src" / "data").is_dir()
            and (candidate / "data" / "splits").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root not found"
    )

PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
GRAPH_DIR = DATA_ROOT / "graphs"

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("Project:", PROJECT_ROOT)

## Xác nhận Per-User Temporal Split trước khi xây dựng graph


In [ ]:
from src.data.preprocessing import validate_splits

SPLIT_DIR = DATA_ROOT / "splits"
with open(SPLIT_DIR / "split_metadata.json", encoding="utf-8") as f:
    split_metadata = json.load(f)

assert split_metadata["split_method"] == "per_user_temporal", (
    f"Expected per_user_temporal, got {split_metadata['split_method']}"
)

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "val.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")
interactions_df = pd.read_csv(DATA_ROOT / "processed" / "interactions.csv")

validate_splits(
    train_df, val_df, test_df, interactions_df,
    split_method="per_user_temporal",
)

print("Split protocol:", split_metadata["split_method"])
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Eligible Validation:", split_metadata["num_eligible_val_users"])
print("Eligible Test:", split_metadata["num_eligible_test_users"])
print("Per-user chronology and split integrity: PASSED")

In [ ]:
from src.data.graph_builder import run_graph_pipeline

metadata = run_graph_pipeline(DATA_ROOT)
assert metadata["split_method"] == "per_user_temporal"

print("Graph construction: PASSED")
display(pd.DataFrame(metadata.items(), columns=["Property", "Value"]))


Đọc hai graph snapshot

In [ ]:
train_graph = torch.load(
    GRAPH_DIR / "train_graph.pt",
    map_location="cpu",
    weights_only=False,
)

train_val_graph = torch.load(
    GRAPH_DIR / "train_val_graph.pt",
    map_location="cpu",
    weights_only=False,
)

print("TRAIN GRAPH")
print(train_graph)

print("\nTRAIN + VALIDATION GRAPH")
print(train_val_graph)

Thống kê node và edge

In [ ]:
node_summary = pd.DataFrame([
    {
        "Node Type": node_type,
        "Count": train_graph[node_type].num_nodes
    }
    for node_type in train_graph.node_types
])

edge_summary = pd.DataFrame([
    {
        "Edge Type": " → ".join(edge_type),
        "Train": train_graph[
            edge_type
        ].edge_index.shape[1],
        "Train + Val": train_val_graph[
            edge_type
        ].edge_index.shape[1],
    }
    for edge_type in train_graph.edge_types
])

display(node_summary)
display(edge_summary)

Biểu đồ so sánh số lượng cạnh

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(edge_summary))
width = 0.35

ax.bar(
    [i - width / 2 for i in x],
    edge_summary["Train"],
    width,
    label="Train"
)

ax.bar(
    [i + width / 2 for i in x],
    edge_summary["Train + Val"],
    width,
    label="Train + Val"
)

ax.set_xticks(list(x))
ax.set_xticklabels(
    edge_summary["Edge Type"],
    rotation=20,
    ha="right"
)

ax.set_title("Heterogeneous Graph Edge Statistics")
ax.set_ylabel("Number of Edges")
ax.legend()

plt.tight_layout()
plt.show()

Kiểm tra graph integrity

In [ ]:
from src.data.graph_builder import (
    validate_graph,
    validate_snapshot_integrity,
)

train = pd.read_csv(
    DATA_ROOT / "splits" / "train.csv"
)

train_val = pd.read_csv(
    DATA_ROOT / "splits" / "train_val.csv"
)

validate_graph(train_graph)
validate_graph(train_val_graph)

validate_snapshot_integrity(
    train_graph,
    train_val_graph,
    train,
    train_val,
)

print("Graph structure: PASSED")
print("Reverse edge consistency: PASSED")
print("Snapshot integrity: PASSED")

## Đối chiếu số cạnh với dữ liệu positive và metadata

In [ ]:
from src.data.graph_builder import RATES

expected_train_edges = int(train_df["is_positive"].sum())
expected_train_val_edges = expected_train_edges + int(val_df["is_positive"].sum())
actual_train_edges = train_graph[RATES].edge_index.shape[1]
actual_train_val_edges = train_val_graph[RATES].edge_index.shape[1]

assert actual_train_edges == expected_train_edges
assert actual_train_val_edges == expected_train_val_edges
assert metadata["train_positive_edges"] == expected_train_edges
assert metadata["train_val_positive_edges"] == expected_train_val_edges
assert train_graph["user"].num_nodes == split_metadata["num_users"]
assert train_graph["movie"].num_nodes == split_metadata["num_movies"]
assert train_graph["genre"].num_nodes == split_metadata["num_genres"]

print("Train positive edges:", actual_train_edges)
print("Train+Validation positive edges:", actual_train_val_edges)
print("Edge counts and node counts: PASSED")

Kiểm tra Test không xuất hiện trong graph

In [ ]:
from src.data.graph_builder import RATES, edge_pairs

test = pd.read_csv(
    DATA_ROOT / "splits" / "test.csv"
)

test_pairs = set(
    map(
        tuple,
        test[["user_idx", "movie_idx"]]
        .to_numpy().tolist(),
    )
)

train_pairs = edge_pairs(train_graph, RATES)
train_val_pairs = edge_pairs(train_val_graph, RATES)

assert train_pairs.isdisjoint(test_pairs)
assert train_val_pairs.isdisjoint(test_pairs)

print("Test interactions excluded: PASSED")

Chạy Unit Tests

In [ ]:
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-p",
        "test_graph_builder.py",
        "-v",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

assert result.returncode == 0

print("Phase 3 unit tests: PASSED")

Lưu báo cáo

In [ ]:
REPORT_DIR = PROJECT_ROOT / "experiments" / "phase3"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

report = {
    "phase": 3,
    "execution": "local",
    "graph_type": "heterogeneous",
    "split_method": split_metadata["split_method"],
    "num_eligible_val_users": split_metadata["num_eligible_val_users"],
    "num_eligible_test_users": split_metadata["num_eligible_test_users"],
    "per_user_chronology": "passed",
    "num_node_types": len(train_graph.node_types),
    "num_edge_types": len(train_graph.edge_types),
    "num_nodes": {
        node_type: train_graph[node_type].num_nodes
        for node_type in train_graph.node_types
    },
    "train_positive_edges": int(
        train_graph[RATES].edge_index.shape[1]
    ),
    "train_val_positive_edges": int(
        train_val_graph[RATES].edge_index.shape[1]
    ),
    "graph_validation": "passed",
    "snapshot_integrity": "passed",
    "test_isolation": "passed",
    "unit_tests": "passed",
}

report_path = REPORT_DIR / "phase3_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Report saved:", report_path)